In [1]:
from albumentations.pytorch import ToTensorV2
from contextlib import nullcontext
from mpl_toolkits.axes_grid1 import make_axes_locatable
from matplotlib.colors import PowerNorm, LinearSegmentedColormap
from torch.utils.data import Dataset, DataLoader
from tqdm.auto import tqdm
from scipy.ndimage import sobel as scipy_sobel, uniform_filter
import torch, torchvision, timm
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import cv2, os, glob, re, math, time, json as _json
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
from matplotlib.patches import Patch

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
#DEVICE = torch.device('cpu')  # GPU ist vom Training belegt
print(f"Device: {DEVICE}")

# =====================================================================
# V3.0 CONFIG
# =====================================================================
CONFIG = {
    'img_height': 480, 'img_width': 640,
    'num_det_classes': 40,
    'num_seg_classes': 6,
    'max_disp_pixel': 192,
    'backbone_stride': 4,
    'internal_disp_steps': 24,
    'tartan_fx': 320.0,
    'tartan_fy': 320.0,
    'tartan_baseline': 0.25,
    'seg_class_weights': [1.0, 3.0, 1.0, 2.0, 2.0, 0.5],
    'deploy': True,    #True -> switch model to deployment mode
}

SEG_CLASS_NAMES = ['WALKABLE', 'STEP', 'WALL', 'OBSTACLE', 'VEGETATION', 'VOID']
SEG_COLORS = np.array([
    [255, 255, 255],    [255, 165, 0],  [100, 100, 200],
    [200, 50, 50],  [0, 150, 0],    [50, 50, 50]
], dtype=np.uint8)

ROBOT_CAT_IDS = [1,2,3,4,6,8,10,11,13,14,15,16,17,18,27,28,31,33,44,47,51,
                  62,63,64,65,67,70,72,73,75,76,77,78,79,81,82,84,85,86,88]
robot_cat_to_continuous = {cid: idx for idx, cid in enumerate(ROBOT_CAT_IDS)}

print(f"V3.1 Eval Config ready")


/home/slarc/miniconda3/envs/stereo_wsl/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Device: cuda
V3.1 Eval Config ready


## Cell 2: V3.1 Model Architecture with deployment adaptations
Paste the **complete** Cell 3 from `FusedHexapodModel_V3_0-Phase-1.ipynb` here.

In [2]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import timm
import math

# --- Hailo-8 Compatible Building Blocks (unchanged from V2.5) ---
class DWSepConv(nn.Module):
    def __init__(self, in_ch, out_ch, kernel_size, stride=1, padding=0, bias=True, dilation=1):
        super().__init__()
        self.dw = nn.Conv2d(in_ch, in_ch, kernel_size, stride=stride, padding=padding,
                            dilation=dilation, groups=in_ch, bias=False)
        self.pw = nn.Conv2d(in_ch, out_ch, 1, bias=bias)
    def forward(self, x):
        return self.pw(self.dw(x))

# --- FPN Neck (unchanged from V2.5) ---
FPN_CH = 64

class LightFPNNeck(nn.Module):
    def __init__(self, ch_s8, ch_s16, ch_s32, fpn_ch=FPN_CH):
        super().__init__()
        self.lat_s8  = nn.Conv2d(ch_s8,  fpn_ch, 1, bias=False)
        self.lat_s16 = nn.Conv2d(ch_s16, fpn_ch, 1, bias=False)
        self.lat_s32 = nn.Conv2d(ch_s32, fpn_ch, 1, bias=False)
        
        # 🚨 PATCH V3.0: RepConv statt DWSepConv für kräftigere Features!
        self.smooth_s8  = RepConv(fpn_ch, fpn_ch)
        self.smooth_s16 = RepConv(fpn_ch, fpn_ch)
        self.bu_s16 = RepConv(fpn_ch, fpn_ch, stride=2)
        self.bu_s32 = RepConv(fpn_ch, fpn_ch, stride=2)

    def forward(self, f_s8, f_s16, f_s32):
        p32 = self.lat_s32(f_s32)
        p16 = self.lat_s16(f_s16) + F.interpolate(p32, scale_factor=2, mode='nearest')
        p8  = self.lat_s8(f_s8) + F.interpolate(p16, scale_factor=2, mode='nearest')
        p8  = self.smooth_s8(p8)
        p16 = self.smooth_s16(p16) + self.bu_s16(p8)
        p32 = p32 + self.bu_s32(p16)
        return p8, p16, p32

class SPPF(nn.Module):
    def __init__(self, c1, c2, k=5):
        super().__init__()
        c_ = c1 // 2  # Hidden Channels
        self.cv1 = nn.Sequential(nn.Conv2d(c1, c_, 1, 1, bias=False), nn.BatchNorm2d(c_), nn.SiLU(inplace=True))
        self.cv2 = nn.Sequential(nn.Conv2d(c_ * 4, c2, 1, 1, bias=False), nn.BatchNorm2d(c2), nn.SiLU(inplace=True))
        self.m = nn.MaxPool2d(kernel_size=k, stride=1, padding=k // 2)

    def forward(self, x):
        x = self.cv1(x)
        y1 = self.m(x)
        y2 = self.m(y1)
        y3 = self.m(y2)
        # Cat von Original + 3 MaxPool-Stufen
        return self.cv2(torch.cat((x, y1, y2, y3), 1))

class ChannelAttention(nn.Module):
    def __init__(self, in_planes, ratio=16):
        super().__init__()
        self.avg_pool = nn.AdaptiveAvgPool2d(1)
        self.max_pool = nn.AdaptiveMaxPool2d(1)
        self.fc1   = nn.Conv2d(in_planes, in_planes // ratio, 1, bias=False)
        self.relu1 = nn.ReLU()
        self.fc2   = nn.Conv2d(in_planes // ratio, in_planes, 1, bias=False)
        self.sigmoid = nn.Sigmoid()

    def forward(self, x):
        avg_out = self.fc2(self.relu1(self.fc1(self.avg_pool(x))))
        max_out = self.fc2(self.relu1(self.fc1(self.max_pool(x))))
        return self.sigmoid(avg_out + max_out)

class SpatialAttention(nn.Module):
    def __init__(self, kernel_size=7):
        super().__init__()
        self.conv1 = nn.Conv2d(2, 1, kernel_size, padding=kernel_size//2, bias=False)
        self.sigmoid = nn.Sigmoid()

    def forward(self, x):
        avg_out = torch.mean(x, dim=1, keepdim=True)
        max_out, _ = torch.max(x, dim=1, keepdim=True)
        x_cat = torch.cat([avg_out, max_out], dim=1)
        return self.sigmoid(self.conv1(x_cat))

class CBAM(nn.Module):
    def __init__(self, in_planes, ratio=16, kernel_size=7):
        super().__init__()
        self.ca = ChannelAttention(in_planes, ratio)
        self.sa = SpatialAttention(kernel_size)

    def forward(self, x):
        x = x * self.ca(x)
        x = x * self.sa(x)
        return x

class GeometryStem(nn.Module):
    def __init__(self, in_ch=24, out_ch=32): # MobileNetV3 s4 hat oft 24 ch
        super().__init__()
        # Zwei Schichten für mehr Reife in den Features
        self.stem = nn.Sequential(
            RepConv(in_ch, out_ch, kernel_size=3, stride=1, padding=1),
            nn.ReLU(inplace=False),
            RepConv(out_ch, out_ch, kernel_size=3, stride=1, padding=1),
            nn.ReLU(inplace=False)
        )

    def forward(self, x):
        return self.stem(x)

import torch
import torch.nn as nn

class AddCoords(nn.Module):
    def __init__(self, h, w, deploy=False):
        super().__init__()
        self.deploy = deploy
        y = torch.linspace(-1, 1, h).view(1, 1, h, 1).expand(1, 1, h, w).contiguous().clone()
        x = torch.linspace(-1, 1, w).view(1, 1, 1, w).expand(1, 1, h, w).contiguous().clone()
        self.register_buffer('y_coords', y)
        self.register_buffer('x_coords', x)

    def forward(self, x_in):
        if self.deploy:
            return torch.cat([x_in, self.y_coords, self.x_coords], dim=1)
        else:
            b = x_in.shape[0]
            return torch.cat([
                x_in,
                self.y_coords.expand(b, -1, -1, -1),
                self.x_coords.expand(b, -1, -1, -1)
            ], dim=1)

class CoordConv2d(nn.Module):
    def __init__(self, in_channels, out_channels, h, w, deploy=False, kernel_size=3, stride=1, padding=1, bias=False):
        super().__init__()
        self.add_coords = AddCoords(h, w, deploy=deploy)
        self.conv = nn.Conv2d(in_channels + 2, out_channels, kernel_size=kernel_size, 
                              stride=stride, padding=padding, bias=bias)
        
    def forward(self, x):
        return self.conv(self.add_coords(x))
    
import torch
import torch.nn as nn
import torch.nn.functional as F

class RepConv(nn.Module):
    """
    Re-Parameterized Convolution:
    Training: 3x3 Conv + 1x1 Conv + Identity (Parallel)
    Inferenz: Eine einzige 3x3 Conv (zusammengefaltet)
    """
    def __init__(self, c1, c2, kernel_size=3, stride=1, padding=1, deploy=False):
        super().__init__()
        self.deploy = deploy
        self.c1 = c1
        self.c2 = c2
        self.stride = stride
        self.padding = padding
        self.act = nn.ReLU(inplace=True)

        if deploy:
            self.rbr_reparam = nn.Conv2d(c1, c2, kernel_size, stride, padding, bias=True)
        else:
            # Identity Branch (nur möglich wenn Dimensionen gleich bleiben)
            self.rbr_identity = nn.BatchNorm2d(c1) if c2 == c1 and stride == 1 else None
            # 3x3 Branch
            self.rbr_dense = nn.Sequential(
                nn.Conv2d(c1, c2, kernel_size, stride, padding, bias=False),
                nn.BatchNorm2d(c2)
            )
            # 1x1 Branch
            self.rbr_1x1 = nn.Sequential(
                nn.Conv2d(c1, c2, 1, stride, 0, bias=False),
                nn.BatchNorm2d(c2)
            )

    def forward(self, x):
        if self.deploy:
            return self.act(self.rbr_reparam(x))
        
        id_out = 0 if self.rbr_identity is None else self.rbr_identity(x)
        return self.act(self.rbr_dense(x) + self.rbr_1x1(x) + id_out)

    def get_equivalent_kernel_bias(self):
        # 3x3 Kernel extrahieren
        kernel3x3, bias3x3 = self._fuse_bn_tensor(self.rbr_dense)
        # 1x1 Kernel extrahieren und auf 3x3 padden
        kernel1x1, bias1x1 = self._fuse_bn_tensor(self.rbr_1x1)
        kernel1x1 = F.pad(kernel1x1, [1, 1, 1, 1])
        # Identity Kernel erstellen (nur 1en in der Mitte)
        kernelid, biasid = self._fuse_bn_tensor(self.rbr_identity)
        
        return kernel3x3 + kernel1x1 + kernelid, bias3x3 + bias1x1 + biasid

    def _fuse_bn_tensor(self, branch):
        if branch is None:
            return torch.zeros((self.c2, self.c1, 3, 3), device=self.rbr_dense[0].weight.device), torch.zeros(self.c2, device=self.rbr_dense[0].weight.device)
        if isinstance(branch, nn.BatchNorm2d):
            # Trick: Fake-Kernel für Identity
            kernel = torch.zeros((self.c1, self.c1, 3, 3), device=branch.weight.device)
            for i in range(self.c1): kernel[i, i, 1, 1] = 1.0
            return self._fuse_bn(kernel, branch.running_mean, branch.running_var, branch.weight, branch.bias, branch.eps)
        else:
            return self._fuse_bn(branch[0].weight, branch[1].running_mean, branch[1].running_var, branch[1].weight, branch[1].bias, branch[1].eps)

    def _fuse_bn(self, kernel, mean, var, gamma, beta, eps):
        std = (var + eps).sqrt()
        t = (gamma / std).reshape(-1, 1, 1, 1)
        return kernel * t, beta - mean * gamma / std
        
    def switch_to_deploy(self):
        if self.deploy: return
        kernel, bias = self.get_equivalent_kernel_bias()
        self.rbr_reparam = nn.Conv2d(self.c1, self.c2, 3, self.stride, self.padding, bias=True)
        self.rbr_reparam.weight.data = kernel
        self.rbr_reparam.bias.data = bias
        # Lösche Trainings-Branches, um RAM zu befreien!
        for attr in ['rbr_dense', 'rbr_1x1', 'rbr_identity']:
            if hasattr(self, attr): delattr(self, attr)
        self.deploy = True
    
# --- Cost Volume (Korrigiert: Universelle Metrik) ---
class CoarseCostVolume(nn.Module):
    def __init__(self, max_disp, in_channels, deploy=False):
        super().__init__()
        self.max_disp = max_disp
        self.deploy = deploy
        self.corr = nn.Conv2d(in_channels * 2, 1, 1, bias=True)
        
    def forward(self, feat_l, feat_r):
        if self.deploy:
            # 🚀 100% STATISCHES ONNX (Batch-Size = 1 impliziert)
            out_slices = []
            
            for d in range(self.max_disp):
                if d == 0:
                    slice_in = torch.cat([feat_l, feat_r], dim=1)
                else:
                    # 🚨 NPU-TRICK: Statt zeros_like + kopieren, nutzen wir Padding!
                    # Wir padden links mit 'd' Nullen und schneiden rechts 'd' Pixel ab.
                    # Das verschiebt das Bild exakt um 'd' Pixel nach rechts!
                    shifted = F.pad(feat_r, pad=(d, 0, 0, 0))[:, :, :, :-d]
                    slice_in = torch.cat([feat_l, shifted], dim=1)
                
                # Faltung direkt anwenden
                out_slice = self.corr(slice_in) 
                out_slices.append(out_slice)
                
            return torch.cat(out_slices, dim=1)
            
        else:
            # 🧠 TRAINING (GPU)
            B, C, H, W = feat_l.shape # Hier ist das Auslesen von B völlig okay!
            cost_slices = []
            for d in range(self.max_disp):
                if d == 0:
                    cost_slices.append(torch.cat([feat_l, feat_r], dim=1))
                else:
                    shifted = torch.zeros_like(feat_r)
                    shifted[:, :, :, d:] = feat_r[:, :, :, :-d]
                    cost_slices.append(torch.cat([feat_l, shifted], dim=1))
            
            cost = torch.stack(cost_slices, dim=2)
            B, C2, D, H, W = cost.shape
            cost = cost.permute(0, 2, 1, 3, 4).reshape(B * D, C2, H, W)
            out = self.corr(cost) 
            return out.view(B, D, H, W)

# --- Refinement Stage with Edge Guidance (from V2.5 Phase 2) ---
class RefinementStage(nn.Module):
    def __init__(self, guidance_channels, scale_factor, use_edge_guidance=False):
        super().__init__()
        self.scale_factor = scale_factor
        self.use_edge_guidance = use_edge_guidance
        extra = 1 if use_edge_guidance else 0
        self.net = nn.Sequential(
            nn.Conv2d(1 + guidance_channels + extra, 32, 3, padding=1), nn.ReLU(inplace=True),
            nn.Conv2d(32, 32, 3, padding=1), nn.ReLU(inplace=True),
            nn.Conv2d(32, 1, 3, padding=1)
        )
        kx = torch.tensor([[-1,0,1],[-2,0,2],[-1,0,1]], dtype=torch.float32).view(1,1,3,3)
        ky = torch.tensor([[-1,-2,-1],[0,0,0],[1,2,1]], dtype=torch.float32).view(1,1,3,3)
        self.register_buffer('kx', kx)
        self.register_buffer('ky', ky)

    def _edge_map(self, img):
        gx = F.conv2d(img, self.kx, padding=1)
        gy = F.conv2d(img, self.ky, padding=1)
        return torch.abs(gx) + torch.abs(gy)       # <-- NEU (L1-Trick)

    # In class RefinementStage(nn.Module):
    def forward(self, disparity_low, guidance, max_disp, gray_img=None): # <--- NEU: max_disp hinzugefügt
        disparity_up = F.interpolate(
            disparity_low, scale_factor=self.scale_factor,
            mode='bilinear', align_corners=False
        ) * self.scale_factor
        
        # ✅ PHYSIKALISCH FUNDIERTE NORMALISIERUNG
        # Wir bringen die Disparität für das CNN auf einen Prozentwert (0.0 bis 1.0)
        norm_disp = disparity_up / max_disp
        
        # Das CNN kriegt jetzt die normierte Disparität + die Guidance-Features
        inp = [norm_disp, guidance]
        
        if self.use_edge_guidance:
            assert gray_img is not None
            if gray_img.shape[-2:] != disparity_up.shape[-2:]:
                gray_img = F.interpolate(gray_img, size=disparity_up.shape[-2:],
                                         mode='bilinear', align_corners=False)
            inp.append(self._edge_map(gray_img))
            
        # Das berechnete Detail-Residual wird zur ORIGINALEN (unskalierten) Disparität addiert!
        return F.relu(disparity_up + self.net(torch.cat(inp, dim=1)))

# --- Stereo Head with Context Network ---
class HierarchicalStereoHead(nn.Module):
    def __init__(self, ch_s8, ch_s4, max_disp_s8, use_normals=True, deploy=False):
        super().__init__()
        self.max_disp_s8 = max_disp_s8
        self.use_normals = use_normals
        self.deploy = deploy
        # ====================================================================
        # 🚨 PATCH V3.0: CoordConv für absolutes räumliches Bewusstsein!
        # kernel_size=1, padding=0 sorgt dafür, dass deine Architektur 
        # exakt gleich bleibt, nur dass X/Y elegant miteingemischt werden.
        # ====================================================================
        
        # 🚨 V3.1 GEOMETRY UPGRADE: 
        # Wir fügen geo_features (32 ch) hinzu. 
        # Für s8 müssen wir sie erst poolen, für s4 passen sie direkt.
        self.reduce_s8 = CoordConv2d(ch_s8 + 32, 32, h=60, w=80, deploy=deploy, kernel_size=1, padding=0, bias=False)


        # S4 Guidance: Normalen (3) + Geo (32) + Backbone (ch_s4)
        s4_guidance_ch = ch_s4 + 32 + (3 if use_normals else 0)
        self.reduce_s4 = CoordConv2d(s4_guidance_ch, 32, h=120, w=160, deploy=deploy, kernel_size=1, padding=0, bias=False)
      
        self.stereo_coarse = CoarseCostVolume(max_disp=self.max_disp_s8, in_channels=32, deploy=deploy)
        self.stereo_refine_s4 = RefinementStage(guidance_channels=32, scale_factor=2.0)
        self.stereo_refine_s1 = RefinementStage(guidance_channels=1, scale_factor=4.0, use_edge_guidance=True)
        
        self.register_buffer('disp_reg', torch.arange(self.max_disp_s8, dtype=torch.float32).view(1, -1, 1, 1))
        self.temperature = 0.7
        self.context_weight = 0.8
        
        # ====================================================================
        # 🚨 PATCH V3.0: Refactoring des Context-Blocks (Fix 2)
        # Sauberer Code, nutzt DWSepConv für die teure mittlere Schicht!
        # ====================================================================
        self.context = nn.Sequential(
            nn.Conv2d(1, 16, 3, padding=1),        # 1 auf 16: normale Conv
            nn.ReLU(inplace=True),
            DWSepConv(16, 16, 3, padding=1),       # 16 auf 16: schlankes DWSepConv!
            nn.ReLU(inplace=True),
            nn.Conv2d(16, 1, 3, padding=1)         # 16 auf 1: Output
        )

        # Lernbare Parameter für GeometryStem
        self.geo_downsample = nn.Conv2d(32, 32, kernel_size=3, stride=2, padding=1, bias=False)

    # 🚨 V3.1: Signatur um geo_features_l und geo_features_r erweitert
    def forward(self, l_s8, r_s8, l_s4, l_img_raw, normals_s4=None, geo_features_l=None, geo_features_r=None):
        
        # --- SICHERHEITSNETZ (Für TensorBoard Tracer / Alte Checkpoints) ---
        if geo_features_l is None:
            B, _, H_s4, W_s4 = l_s4.shape
            geo_features_l = torch.zeros(B, 32, H_s4, W_s4, device=l_s4.device, dtype=l_s4.dtype)
        if geo_features_r is None:
            B, _, H_s4, W_s4 = l_s4.shape
            geo_features_r = torch.zeros(B, 32, H_s4, W_s4, device=l_s4.device, dtype=l_s4.dtype)

        # --- STUFE 1: S8 Coarse Matching ---
        # Da geo_features auf s4 (z.B. 160x120) sind, poolen wir sie für s8 (80x60)
        #geo_l_s8 = F.avg_pool2d(geo_features_l, kernel_size=2, stride=2)
        #geo_r_s8 = F.avg_pool2d(geo_features_r, kernel_size=2, stride=2)
        
        # geo_downsample statt avg_pool2d, um die Kantenschärfe zu erhalten
        geo_l_s8 = self.geo_downsample(geo_features_l)
        geo_r_s8 = self.geo_downsample(geo_features_r)
        
        # Jetzt mit Backbone-Features mischen (+ 32 Kanäle!)
        feat_l_s8 = self.reduce_s8(torch.cat([l_s8, geo_l_s8], dim=1))
        feat_r_s8 = self.reduce_s8(torch.cat([r_s8, geo_r_s8], dim=1))
        
        # --- STUFE 2: S4 Refinement Guidance ---
        if self.use_normals:
            if normals_s4 is not None:
                norm_in = normals_s4
            else:
                norm_in = torch.zeros(l_s4.size(0), 3, l_s4.size(2), l_s4.size(3), device=l_s4.device, dtype=l_s4.dtype)
            
            # Alle drei Quellen: Backbone (s4) + GeoStem (s4) + Normals (s4)
            l_s4_combined = torch.cat([l_s4, geo_features_l, norm_in], dim=1)
        else:
            l_s4_combined = torch.cat([l_s4, geo_features_l], dim=1)
            
        feat_l_s4 = self.reduce_s4(l_s4_combined)
        
        # --- STUFE 3: Stereo Prozess ---
        vol_s8 = self.stereo_coarse(feat_l_s8, feat_r_s8)
        
        # Kontext-Netzwerk
        if self.deploy:
            # 🚀 STATISCH FÜR ONNX (Kein .shape Aufruf mehr!)
            ctx_slices = []
            
            # Wir nutzen die harte Konstante self.max_disp_s8 (z.B. 48)
            for d in range(self.max_disp_s8):
                # Wir schneiden einfach fest den jeweiligen Kanal heraus
                d_slice = vol_s8[:, d:d+1, :, :]
                ctx_slice = self.context(d_slice)
                ctx_slices.append(ctx_slice)
                
            vol_ctx = torch.cat(ctx_slices, dim=1)
            
        else:
            # 🧠 TRAINING FÜR GPU (mit Batch-Logik)
            B, D, H, W = vol_s8.shape
            vol_reshaped = vol_s8.view(B * D, 1, H, W)
            vol_ctx = self.context(vol_reshaped).view(B, D, H, W)
        
        vol_s8 = self.context_weight * vol_ctx + (1.0 - self.context_weight) * vol_s8
        
        prob_s8 = F.softmax(vol_s8 / self.temperature, dim=1)
        disp_s8 = torch.sum(prob_s8 * self.disp_reg, dim=1, keepdim=True)
        
        max_disp_s4 = self.max_disp_s8 * 2.0
        disp_s4 = self.stereo_refine_s4(disp_s8, feat_l_s4, max_disp=max_disp_s4)
        
        max_disp_s1 = max_disp_s4 * 4.0
        final_disp = self.stereo_refine_s1(disp_s4, l_img_raw, max_disp=max_disp_s1, gray_img=l_img_raw)
        
        return final_disp, disp_s8

# ✅ V3.0 Normals Head - Multi-Scale Fusion (s4 + s8), Up-Sampling, Image-Guided Refinement, CoordConv
class NormalsHead(nn.Module):
    def __init__(self, ch_s4, ch_s8, deploy=False): # NEU: deploy flag
        super().__init__()
        self.deploy = deploy
        # Stage 1: Multi-Scale Fusion (s4 + s8)
        self.s8_adapt = nn.Conv2d(ch_s8, 32, kernel_size=1)
        
        # 🚨 V3.1 FIX: Backbone (ch_s4) + s8_adapt (32) + geo_features (32)
        fused_ch = ch_s4 + 32 + 32 
        
        # ====================================================================
        # 🚨 PATCH V3.0: CoordConv für den NormalsHead!
        # ====================================================================
        self.stage1 = nn.Sequential(
            CoordConv2d(fused_ch, 96, h=120, w=160, deploy=deploy, kernel_size=3, padding=1), 
            nn.ReLU(inplace=True),
            nn.Conv2d(96, 64, 3, padding=1), nn.ReLU(inplace=True),
            nn.Conv2d(64, 32, 3, padding=1), nn.ReLU(inplace=True)
        )
        self.coarse_out = nn.Conv2d(32, 3, 3, padding=1)

        # Stage 3: Image-Guided Refinement
        # Inputs: 3 (Coarse Normals) + 1 (Gray Img) + 1 (Sobel Edges) = 5
        self.refiner = nn.Sequential(
            nn.Conv2d(5, 32, 3, padding=1),
            
            # ====================================================================
            # 🚨 PATCH V3.0: Clean Code mit DWSepConv! 
            # (Ersetzt die 4 manuellen Layer von vorher)
            # ====================================================================
            DWSepConv(32, 32, 3, padding=1),
            nn.ReLU(inplace=True),
            
            DWSepConv(32, 32, 3, padding=1),
            nn.ReLU(inplace=True),
            
            nn.Conv2d(32, 3, 3, padding=1)
        )
        
        # Fest verdrahtete Sobel-Filter (keine trainierbaren Parameter)
        sobel_x = torch.tensor([[-1., 0., 1.], [-2., 0., 2.], [-1., 0., 1.]]).view(1, 1, 3, 3)
        sobel_y = torch.tensor([[-1., -2., -1.], [0., 0., 0.], [1., 2., 1.]]).view(1, 1, 3, 3)
        self.register_buffer('sobel_x', sobel_x)
        self.register_buffer('sobel_y', sobel_y)

    def get_edges(self, img):
        gx = F.conv2d(img, self.sobel_x, padding=1)
        gy = F.conv2d(img, self.sobel_y, padding=1)
        # 🚨 100% Hailo-Safe: L1-Norm (Manhattan-Distanz) statt Wurzel
        # Das löst auch das FP16 NaN-Problem automatisch, da es keine Wurzel mehr gibt!
        return torch.abs(gx) + torch.abs(gy)

    def forward(self, l_s4, l_s8, gray_img, geo_features=None):
        # 1. s8 Features anpassen und auf s4 Größe hochziehen
        s8_adapted = self.s8_adapt(l_s8)
        s8_adapted = F.interpolate(s8_adapted, size=l_s4.shape[2:], mode='bilinear', align_corners=False)
        
        # 2. 🚨 V3.1 Alle drei Quellen zusammenbauen!
        if geo_features is not None:
            l_s4_combined = torch.cat([l_s4, s8_adapted, geo_features], dim=1)
        else:
            # 🚨 FIX: Hartkodierte 32 Kanäle für den Dummy-Tensor, falls TensorBoard testet
            dummy_geo = torch.zeros(l_s4.size(0), 32, l_s4.size(2), l_s4.size(3), 
                                    device=l_s4.device, dtype=l_s4.dtype)
            l_s4_combined = torch.cat([l_s4, s8_adapted, dummy_geo], dim=1)
        
        # 3. Ab in den CoordConv
        feat_s4 = self.stage1(l_s4_combined)
        
        # Normale berechnen (Coarse)
        coarse_normals_s4 = self.coarse_out(feat_s4)
        if not self.deploy:
            coarse_normals_s4 = F.normalize(coarse_normals_s4, dim=1)
        
        # Upsample für Stage 3
        normals_coarse = F.interpolate(coarse_normals_s4, size=gray_img.shape[2:], mode='bilinear', align_corners=False)
        
        # Stage 3: Image-Guided Refinement
        edges = self.get_edges(gray_img)
        refine_in = torch.cat([normals_coarse, gray_img, edges], dim=1)
        refined = self.refiner(refine_in)
        
        # Residual-Verbindung & Normalisierung
        normals_s1 = normals_coarse + refined
        # 🚨 FP16-Safe Epsilon explizit setzen (Standard ist 1e-12 -> in FP16 ist das 0.0 -> NaN!)
        if self.deploy:
            # 🚀 DEPLOYMENT: Nackte Convolution-Outputs ausgeben (Hailo liebt das)
            return normals_s1, coarse_normals_s4
        else:
            # 🧠 TRAINING: L2-Normalisierung für die Loss-Funktion
            normals_s1 = F.normalize(normals_s1, p=2, dim=1, eps=1e-4)
            normals_s4 = F.normalize(coarse_normals_s4, p=2, dim=1, eps=1e-4)
            return normals_s1, normals_s4

# ✅ V3.1: LRASPPHead with Normals input + 6 classes
class LRASPPHead(nn.Module):
    def __init__(self, low_ch, high_ch, num_classes, normals_ch=3):
        super().__init__()
        self.cbr_high = nn.Sequential(
            nn.Conv2d(high_ch, 128, 1, bias=False), nn.BatchNorm2d(128), nn.ReLU(inplace=True)
        )
        self.scale_high = nn.Sequential(
            nn.AvgPool2d(kernel_size=(30, 40)),
            nn.Conv2d(high_ch, 128, 1, bias=False),
            nn.Sigmoid()
        )
        # ✅ V2.9: low_classifier takes backbone features + predicted normals
        self.low_classifier = nn.Conv2d(low_ch + normals_ch, num_classes, 1)
        self.high_classifier = nn.Conv2d(128, num_classes, 1)
        # Dilated conv also gets normals (activated in Phase 2)
        self.mid_classifier = nn.Conv2d(128 + normals_ch, num_classes, 3, padding=2, dilation=2)
        self.use_mid = True
        
    def forward(self, x_low, x_high, normals_s4=None):
        out = self.cbr_high(x_high) * self.scale_high(x_high)
        out = F.interpolate(out, scale_factor=4.0, mode='bilinear', align_corners=False)
        
        if normals_s4 is not None:
            low_in = torch.cat([x_low, normals_s4], dim=1)
        else:
            # Make it Hailo compatible (kein F.pad auf Channels):
            dummy_normals = torch.zeros(x_low.size(0), 3, x_low.size(2), x_low.size(3), device=x_low.device)
            low_in = torch.cat([x_low, dummy_normals], dim=1)
            
        result = self.low_classifier(low_in) + self.high_classifier(out)
        
        if self.use_mid:
            if normals_s4 is not None:
                mid_in = torch.cat([out, normals_s4], dim=1)
            else:
                # ✅ KORRIGIERT: Auch hier dummy_normals mit torch.cat statt F.pad
                dummy_normals_mid = torch.zeros(out.size(0), 3, out.size(2), out.size(3), device=out.device)
                mid_in = torch.cat([out, dummy_normals_mid], dim=1)
                
            result = result + self.mid_classifier(mid_in)   
        return result
        
# --- YOLO Heads (unchanged structure, 40 classes) ---
class DecoupledHead(nn.Module):
    def __init__(self, ch_in, num_classes, h, w, deploy=False, width=128):
        super().__init__()
        
        self.coord_conv_cls = CoordConv2d(ch_in, width, h=h, w=w, deploy=deploy, kernel_size=3, padding=1)
        self.coord_conv_reg = CoordConv2d(ch_in, width, h=h, w=w, deploy=deploy, kernel_size=3, padding=1)
        
        # 🚨 PATCH V3.0: RepConv integriert die ReLU bereits intern!
        self.cls_convs = nn.Sequential(
            self.coord_conv_cls,
            nn.BatchNorm2d(width),
            nn.ReLU(inplace=True),
            RepConv(width, width) # ⬅️ RepConv statt DWSepConv
        )
        
        self.reg_convs = nn.Sequential(
            self.coord_conv_reg,
            nn.BatchNorm2d(width),
            nn.ReLU(inplace=True),
            RepConv(width, width) # ⬅️ RepConv statt DWSepConv
        )
        
        self.cls_pred = nn.Conv2d(width, num_classes, 1)
        self.reg_pred = nn.Conv2d(width, 4, 1)
        self.obj_pred = nn.Conv2d(width, 1, 1)
    def forward(self, x):
        cls_feat = self.cls_convs(x); reg_feat = self.reg_convs(x)
        return torch.cat([self.reg_pred(reg_feat), self.obj_pred(reg_feat), self.cls_pred(cls_feat)], dim=1)

class YOLOHead(nn.Module):
    def __init__(self, fpn_ch=FPN_CH, num_classes=40, deploy=False):
        super().__init__()
        self.head_s8  = DecoupledHead(fpn_ch, num_classes, h=60, w=80, deploy=deploy, width=128)
        self.head_s16 = DecoupledHead(fpn_ch, num_classes, h=30, w=40, deploy=deploy, width=128)
        self.head_s32 = DecoupledHead(fpn_ch, num_classes, h=15, w=20, deploy=deploy, width=128)
    def forward(self, x_s8, x_s16, x_s32):
        return [self.head_s8(x_s8), self.head_s16(x_s16), self.head_s32(x_s32)]

# =====================================================================
# ✅ V3.1: FusedHexapodModel — 1-Channel Input, 5 Outputs
# =====================================================================
class FusedHexapodModel(nn.Module):
    def __init__(self, config):
        super().__init__()
        self.deploy_mode = config.get('deploy', False)
        self.backbone = timm.create_model('mobilenetv3_large_100', pretrained=True,
                                           features_only=True, out_indices=(1, 2, 3, 4))
        feat_info = self.backbone.feature_info.channels()
        ch_s4, ch_s8, ch_s16, ch_s32 = feat_info

        # ✅ V2.9: Patch first conv to 1-channel input
        old_conv = self.backbone.conv_stem
        new_conv = nn.Conv2d(1, old_conv.out_channels, kernel_size=old_conv.kernel_size,
                              stride=old_conv.stride, padding=old_conv.padding, bias=False)
        # Merge RGB weights via luminance formula
        with torch.no_grad():
            w = old_conv.weight.data  # [C_out, 3, kH, kW]
            new_conv.weight.data = w[:, 0:1]*0.299 + w[:, 1:2]*0.587 + w[:, 2:3]*0.114
        self.backbone.conv_stem = new_conv
        print(f"  ✅ Backbone first conv: 3→1 channel (luminance merge)")
        
        # 🚨 PATCH V3.0: SPPF initialisieren (nimmt ch_s32 dynamisch von timm!)
        self.sppf = SPPF(c1=ch_s32, c2=ch_s32, k=5)
        print(f"  ✅ SPPF Module injected at s32 ({ch_s32} channels)")
        
        disp_steps = config.get('internal_disp_steps', 48)
        self.stereo_head = HierarchicalStereoHead(ch_s8, ch_s4, max_disp_s8=disp_steps, deploy=self.deploy_mode)
        self.normals_head = NormalsHead(ch_s4, ch_s8, deploy=self.deploy_mode)
        self.seg_head = LRASPPHead(ch_s4, ch_s16, config['num_seg_classes'], normals_ch=3)
        self.fpn_neck = LightFPNNeck(ch_s8, ch_s16, ch_s32, fpn_ch=FPN_CH)

        # 🚨 PATCH V3.0: CBAM Attention für die FPN-Outputs
        self.cbam_s8  = CBAM(FPN_CH)
        self.cbam_s16 = CBAM(FPN_CH)
        self.cbam_s32 = CBAM(FPN_CH)
        print(f"  ✅ CBAM Attention Modules injected after FPN")

        self.geo_stem = GeometryStem(in_ch=24, out_ch=32)
        print(f"  ✅ Geometry Stem Modules injected after Backbone")

        self.yolo_head = YOLOHead(fpn_ch=FPN_CH, num_classes=config['num_det_classes'], deploy=self.deploy_mode)

        total_params = sum(p.numel() for p in self.parameters())
        print(f"  Total parameters: {total_params:,}")
        print(f"  Normals Head: {sum(p.numel() for p in self.normals_head.parameters()):,}")
        print(f"  Seg Head: {sum(p.numel() for p in self.seg_head.parameters()):,}")
        print(f"  YOLO Head: {sum(p.numel() for p in self.yolo_head.parameters()):,}")

    def forward(self, x_left, x_right, use_normals_for_stereo=False):
        # Backbone & FPN
        features_l = self.backbone(x_left)
        # ====================================================================
        # 🚨 NEU V3.1: GEOMETRY STEM (High-Res Pfad)
        # Nutzt features_l[0] (s4 / 160x120), um scharfe Kanten-Features zu extrahieren
        # ====================================================================
        geo_feat_l = self.geo_stem(features_l[0])
        # Beide Auflösungen vom Normals-Head abgreifen (Neu: mit s8 und gray_img)
        # Normals bekommt jetzt die Geo-Features zusätzlich
        normals_s1, normals_s4 = self.normals_head(
            features_l[0], 
            features_l[1], 
            x_left, 
            geo_features=geo_feat_l # 👈 NEU: High-Res Support
        )
        
        final_disp, disp_s8 = None, None
        
        # ✅ FIX: Nur Stereo ausführen, wenn wir auch ein rechtes Bild haben (TartanAir)
        if x_right is not None:
            with torch.no_grad():
                features_r = self.backbone(x_right)
                # Auch für das rechte Bild brauchen wir die Geo-Features für das Matching!
                geo_feat_r = self.geo_stem(features_r[0])
            
            # 🚨 FIX: normals_s4.detach() verhindert, dass Stereo-Gradienten den NormalsHead zerstören!
            if not self.deploy_mode:
                normals_s4_for_stereo = normals_s4.detach() if (use_normals_for_stereo and normals_s4 is not None) else None
            else:
                normals_s4_for_stereo = normals_s4
            
            # Stereo Head (nutzt jetzt die entkoppelten Normalen UND das Graustufenbild)
            # Stereo Head bekommt jetzt geo_feat_l UND geo_feat_r
            final_disp, disp_s8 = self.stereo_head(
                features_l[1],    # l_s8
                features_r[1],    # r_s8
                features_l[0],    # l_s4
                x_left,           # l_img_raw
                normals_s4=normals_s4_for_stereo,
                geo_features_l=geo_feat_l, # 👈 NEU: Linke Geo-Features
                geo_features_r=geo_feat_r  # 👈 NEU: Rechte Geo-Features
            )
        else:
            final_disp, disp_s8 = None, None
            
        # Seg bekommt ebenfalls die S4-Normalen (160x120)
        normals_for_others = normals_s4.detach() if (use_normals_for_stereo and normals_s4 is not None) else None
        seg = self.seg_head(features_l[0], features_l[2], normals_s4=normals_for_others)

        # 🚨 PATCH V3.0: SPPF auf die tiefste Ebene anwenden
        f_s32_sppf = self.sppf(features_l[3])
        # FPN_Neck mit der gepatchten s32-Map aufrufen
        fpn_s8, fpn_s16, fpn_s32 = self.fpn_neck(features_l[1], features_l[2], f_s32_sppf)
        
        # ====================================================================
        # 🚨 PATCH V3.0: CBAM Attention vor dem YOLO-Head anwenden!
        # ====================================================================
        fpn_s8_att  = self.cbam_s8(fpn_s8)
        fpn_s16_att = self.cbam_s16(fpn_s16)
        fpn_s32_att = self.cbam_s32(fpn_s32)

        # YOLO bekommt jetzt die gefilterten Attention-Features!
        det = self.yolo_head(fpn_s8_att, fpn_s16_att, fpn_s32_att)

        if not self.deploy_mode:
            return final_disp, seg, det, disp_s8, normals_s1
        else:
            yolo_s8, yolo_s16, yolo_s32 = det
            return final_disp, seg, disp_s8, normals_s1, yolo_s8, yolo_s16, yolo_s32




In [3]:
import torch
import os
import copy

# =====================================================================
# 🛠️ KONFIGURATION
# =====================================================================
checkpoint_paths = [
    "/home/slarc/jupyter/checkpoints/checkpoint_v3_1_best.pth",
    "/home/slarc/jupyter/checkpoints/checkpoint_v3_1_step_59136.pth",
    "/home/slarc/jupyter/checkpoints/checkpoint_v3_1_step_57904.pth",
    "/home/slarc/jupyter/checkpoints/checkpoint_v3_1_step_56672.pth",
    "/home/slarc/jupyter/checkpoints/checkpoint_v3_1_step_55440.pth",
    "/home/slarc/jupyter/checkpoints/checkpoint_v3_1_step_54208.pth"
]

# 1. Config auf Deploy stellen (Schaltet die Normalisierung ab!)
CONFIG['deploy'] = True
output_path_folded = "checkpoints/checkpoint_v3_1_deploy.pth"

# =====================================================================
# 🚀 SWA + FOLDING LOGIK
# =====================================================================
def create_swa_and_fold(model, filepaths, out_path):
    print(f"🔄 1. Starte SWA: Mittle {len(filepaths)} Checkpoints...")
    
    swa_state_dict = None
    num_checkpoints = len(filepaths)
    
    for path in filepaths:
        if not os.path.exists(path):
            raise FileNotFoundError(f"❌ Checkpoint nicht gefunden: {path}")
            
        print(f"📥 Lade: {path}")
        # Alles auf die CPU laden für den SWA-Prozess (spart VRAM)
        ckpt = torch.load(path, map_location='cpu')
        model_state = ckpt.get('model_state_dict', ckpt)
        
        if swa_state_dict is None:
            swa_state_dict = {k: v.clone() for k, v in model_state.items()}
        else:
            for k in swa_state_dict.keys():
                if k in model_state:
                    swa_state_dict[k] += model_state[k]
    
    # Durchschnitt berechnen
    for k in swa_state_dict.keys():
        if swa_state_dict[k].is_floating_point():
            swa_state_dict[k].div_(num_checkpoints)
        else:
            swa_state_dict[k] = torch.div(swa_state_dict[k], num_checkpoints, rounding_mode='floor')
            
    print("✅ Mittelung abgeschlossen.")
    
    # -----------------------------------------------------------------
    # 🏗️ MODELL LADEN UND FALTEN (switch_to_deploy)
    # -----------------------------------------------------------------
    print("🏗️ 2. Lade SWA-Gewichte in Modellstruktur...")
    missing, unexpected = model.load_state_dict(swa_state_dict, strict=False)
    # Sicherheitscheck: nur Buffer sollten fehlen, keine Gewichte!
    unexpected_weights = [k for k in missing if 'coords' not in k]
    if unexpected_weights:
        raise RuntimeError(f"❌ Unerwartete fehlende Gewichte: {unexpected_weights}")
    else:
        print(f"✅ Gewichte geladen. {len(missing)} CoordConv-Buffer werden neu initialisiert (erwartet).")
    model.eval() # WICHTIG: BatchNorm einfrieren vor dem Falten!
    
    print("🧬 3. Rufe switch_to_deploy() auf (RepConv Faltung)...")
    folded_count = 0
    for m in model.modules():
        if hasattr(m, 'switch_to_deploy'):
            m.switch_to_deploy()
            folded_count += 1
            
    print(f"✨ {folded_count} Layer erfolgreich gefaltet!")
    
    # 💾 Gefaltetes PyTorch Modell als Backup speichern
    os.makedirs(os.path.dirname(out_path), exist_ok=True)
    torch.save(model.state_dict(), out_path)
    print(f"💾 Gefaltetes SWA-Modell gespeichert unter: {out_path}")
    
    return model

# =====================================================================
# 🏁 AUSFÜHRUNG & ONNX EXPORT
# =====================================================================

# 2. Modell-Hülle erstellen (auf CPU für RAM-Effizienz während SWA)
model = FusedHexapodModel(CONFIG)

# 3. SWA ausführen und Modell falten
model = create_swa_and_fold(model, checkpoint_paths, output_path_folded)

# 4. Modell für den ONNX-Export auf die GPU schieben
model = model.to(DEVICE)
dummy_l = torch.randn(1, 1, 480, 640).to(DEVICE)
dummy_r = torch.randn(1, 1, 480, 640).to(DEVICE)

# 5. Sauberer ONNX Export
onnx_path = "hexapod_v3_1_deploy.onnx"
print(f"🚀 Exportiere sauberes Modell nach {onnx_path}...")

torch.onnx.export(
    model,
    (dummy_l, dummy_r),
    onnx_path,
    input_names=['input_layer1', 'input_layer2'],
    output_names=['disp_final', 'seg', 'disp_s8', 'normals', 'yolo_s8', 'yolo_s16', 'yolo_s32'],
    opset_version=13,
    do_constant_folding=True
)

print("🎉 Export abgeschlossen! Dieses Modell ist jetzt zu 100% Hailo-Ready.")

Unexpected keys (classifier.bias, classifier.weight, conv_head.bias, conv_head.weight) found while loading pretrained weights. This may be expected if model is being adapted.
/tmp/ipykernel_89622/4054521527.py:36: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file.

  ✅ Backbone first conv: 3→1 channel (luminance merge)
  ✅ SPPF Module injected at s32 (960 channels)
  ✅ CBAM Attention Modules injected after FPN
  ✅ Geometry Stem Modules injected after Backbone
  Total parameters: 7,237,261
  Normals Head: 158,790
  Seg Head: 36,950
  YOLO Head: 1,462,791
🔄 1. Starte SWA: Mittle 6 Checkpoints...
📥 Lade: /home/slarc/jupyter/checkpoints/checkpoint_v3_1_best.pth
📥 Lade: /home/slarc/jupyter/checkpoints/checkpoint_v3_1_step_59136.pth
📥 Lade: /home/slarc/jupyter/checkpoints/checkpoint_v3_1_step_57904.pth
📥 Lade: /home/slarc/jupyter/checkpoints/checkpoint_v3_1_step_56672.pth
📥 Lade: /home/slarc/jupyter/checkpoints/checkpoint_v3_1_step_55440.pth
📥 Lade: /home/slarc/jupyter/checkpoints/checkpoint_v3_1_step_54208.pth
✅ Mittelung abgeschlossen.
🏗️ 2. Lade SWA-Gewichte in Modellstruktur...
✅ Gewichte geladen. 30 CoordConv-Buffer werden neu initialisiert (erwartet).
🧬 3. Rufe switch_to_deploy() auf (RepConv Faltung)...
✨ 12 Layer erfolgreich gefaltet!
💾 Gefalte

/tmp/ipykernel_89622/116824965.py:308: TracerWarning: Converting a tensor to a Python boolean might cause the trace to be incorrect. We can't record the data flow of Python values, so this value will be treated as a constant in the future. This means that the trace might not generalize to other inputs!
  if gray_img.shape[-2:] != disparity_up.shape[-2:]:
/tmp/ipykernel_89622/116824965.py:710: TracerWarning: Converting a tensor to a Python boolean might cause the trace to be incorrect. We can't record the data flow of Python values, so this value will be treated as a constant in the future. This means that the trace might not generalize to other inputs!
  normals_for_others = normals_s4.detach() if (use_normals_for_stereo and normals_s4 is not None) else None
/home/slarc/miniconda3/envs/stereo_wsl/lib/python3.10/site-packages/torch/onnx/_internal/jit_utils.py:308: UserWarning: Constant folding - Only steps=1 can be constant folded for opset >= 10 onnx::Slice op. Constant folding not app

🎉 Export abgeschlossen! Dieses Modell ist jetzt zu 100% Hailo-Ready.


In [4]:
import onnx

model_onnx = onnx.load("hexapod_v3_1_deploy.onnx")
onnx.checker.check_model(model_onnx)
print("✅ ONNX-Graph ist valide")
print(f"Opset: {model_onnx.opset_import[0].version}")

# Inputs/Outputs anzeigen
for inp in model_onnx.graph.input:
    print(f"Input:  {inp.name} — {[d.dim_value for d in inp.type.tensor_type.shape.dim]}")
for out in model_onnx.graph.output:
    print(f"Output: {out.name} — {[d.dim_value for d in out.type.tensor_type.shape.dim]}")



✅ ONNX-Graph ist valide
Opset: 13
Input:  input_layer1 — [1, 1, 480, 640]
Input:  input_layer2 — [1, 1, 480, 640]
Output: disp_final — [1, 1, 480, 640]
Output: seg — [1, 6, 120, 160]
Output: disp_s8 — [1, 1, 60, 80]
Output: normals — [1, 3, 480, 640]
Output: yolo_s8 — [1, 45, 60, 80]
Output: yolo_s16 — [1, 45, 30, 40]
Output: yolo_s32 — [1, 45, 15, 20]


In [5]:
import onnxruntime as ort
import numpy as np
import torch

# Gleiche Dummy-Inputs
dummy_l = torch.randn(1, 1, 480, 640)
dummy_r = torch.randn(1, 1, 480, 640)

# PyTorch-Output
model.cpu().eval()
with torch.no_grad():
    pt_out = model(dummy_l, dummy_r)

# ONNX-Output
sess = ort.InferenceSession("hexapod_v3_1_deploy.onnx")
onnx_out = sess.run(None, {
    'input_layer1': dummy_l.numpy(),
    'input_layer2': dummy_r.numpy()
})

# Vergleich pro Output
names = ['disp_final', 'seg', 'disp_s8', 'normals', 'yolo_s8', 'yolo_s16', 'yolo_s32']
for i, name in enumerate(names):
    pt = pt_out[i].numpy() if i < 2 else pt_out[i].numpy()
    diff = np.abs(pt - onnx_out[i]).max()
    print(f"{name}: max_diff = {diff:.6f} {'✅' if diff < 1e-4 else '⚠️'}")

disp_final: max_diff = 0.001247 ⚠️
seg: max_diff = 0.000051 ✅
disp_s8: max_diff = 0.000392 ⚠️
normals: max_diff = 0.000037 ✅
yolo_s8: max_diff = 0.000059 ✅
yolo_s16: max_diff = 0.000048 ✅
yolo_s32: max_diff = 0.000041 ✅


In [6]:
!pip install onnxsim
!python -m onnxsim hexapod_v3_1_deploy.onnx hexapod_v3_1_simplified.onnx --no-large-tensor

Simplifying...
Finish! Here is the difference:
┏━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━┓
┃                   ┃ Original Model ┃ Simplified Model ┃
┡━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━┩
│ Abs               │ 4              │ 2                │
│ Add               │ 29             │ 28               │
│ AveragePool       │ 1              │ 1                │
│ Cast              │ 23             │ 0                │
│ Concat            │ 73             │ 45               │
│ Constant          │ 652            │ 340              │
│ ConstantOfShape   │ 23             │ 0                │
│ Conv              │ 286            │ 284              │
│ Div               │ 3              │ 3                │
│ GlobalAveragePool │ 3              │ 3                │
│ HardSigmoid       │ 32             │ 32               │
│ Identity          │ 48             │ 0                │
│ MaxPool           │ 6              │ 6                │
│ Mul               │ 46 

In [7]:
import torch

# 1. Modul initialisieren (mit deinen Parametern)
max_disp = 48
in_channels = 32
model = CoarseCostVolume(max_disp=max_disp, in_channels=in_channels)
model.eval() # Sehr wichtig: Dropout & BatchNorm einfrieren!

# 2. Realistische Dummy-Daten erzeugen (Batch=1 für Deployment-Test)
B, C, H, W = 1, 32, 60, 80
feat_l = torch.randn(B, C, H, W)
feat_r = torch.randn(B, C, H, W)

print("🧪 Starte mathematischen Äquivalenz-Test...")

with torch.no_grad():
    # --- DURCHLAUF 1: Alter Trainings-Code (GPU-optimiert) ---
    model.deploy = False
    out_training = model(feat_l, feat_r)
    print(f"✅ Training-Output Shape: {out_training.shape}")

    # --- DURCHLAUF 2: Neuer NPU-Code (Hailo-optimiert) ---
    model.deploy = True
    out_deploy = model(feat_l, feat_r)
    print(f"✅ Deploy-Output Shape:   {out_deploy.shape}")

# 3. Numerischer Vergleich
# Wir berechnen den maximalen Unterschied zwischen beiden Tensoren
max_diff = torch.abs(out_training - out_deploy).max().item()

print("-" * 40)
print(f"📊 Maximale Abweichung: {max_diff:.10f}")

# Eine Abweichung von < 1e-6 (0.000001) gilt bei Float32 als "perfekt identisch"
if max_diff < 1e-6:
    print("🎉 ERFOLG: Die Pfade sind mathematisch zu 100% identisch!")
else:
    print("⚠️ FEHLER: Es gibt eine Abweichung.")

🧪 Starte mathematischen Äquivalenz-Test...
✅ Training-Output Shape: torch.Size([1, 48, 60, 80])
✅ Deploy-Output Shape:   torch.Size([1, 48, 60, 80])
----------------------------------------
📊 Maximale Abweichung: 0.0000000000
🎉 ERFOLG: Die Pfade sind mathematisch zu 100% identisch!


In [8]:
model_onnx = onnx.load("hexapod_v3_1_simplified.onnx")
onnx.checker.check_model(model_onnx)
for out in model_onnx.graph.output:
    print(f"Output: {out.name} — {[d.dim_value for d in out.type.tensor_type.shape.dim]}")

Output: disp_final — [1, 1, 480, 640]
Output: seg — [1, 6, 120, 160]
Output: disp_s8 — [1, 1, 60, 80]
Output: normals — [1, 3, 480, 640]
Output: yolo_s8 — [1, 45, 60, 80]
Output: yolo_s16 — [1, 45, 30, 40]
Output: yolo_s32 — [1, 45, 15, 20]


In [9]:
import numpy as np
import cv2, glob, random, os

random.seed(42)

TARTAN_TRAIN_ROOT = "/home/slarc/datasets/TartanAir"
N_SAMPLES = 1024

# Alle TartanAir Bildpaare sammeln
samples = []
for env in sorted(glob.glob(os.path.join(TARTAN_TRAIN_ROOT, '*'))):
    for diff in ['Easy', 'Hard']:
        for traj in sorted(glob.glob(os.path.join(env, diff, 'P*'))):
            left_dir = os.path.join(traj, 'image_left')
            right_dir = os.path.join(traj, 'image_right')
            if not all(os.path.exists(d) for d in [left_dir, right_dir]):
                continue
            for lp in sorted(glob.glob(os.path.join(left_dir, '*.png'))):
                fn = os.path.basename(lp).replace('_left.png', '')
                rp = os.path.join(right_dir, fn + '_right.png')
                if os.path.exists(rp):
                    samples.append({'l': lp, 'r': rp})

print(f"✅ {len(samples)} Bildpaare gefunden")
pairs = random.sample(samples, min(N_SAMPLES, len(samples)))

calib_left, calib_right = [], []
for s in pairs:
    for path, out in [(s['l'], calib_left), (s['r'], calib_right)]:
        img = cv2.imread(path)
        img = cv2.resize(img, (640, 480))
        gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY).astype(np.float32)
        # Keine Normalisierung! Das übernimmt jetzt das .alls
        out.append(gray[np.newaxis])  # [1, H, W], float32, 0-255

np.save('calib_left.npy', np.array(calib_left))
np.save('calib_right.npy', np.array(calib_right))
print(f"✅ calib_left.npy / calib_right.npy gespeichert — {len(pairs)} Paare, range [0, 255]")

✅ 84824 Bildpaare gefunden
✅ calib_left.npy / calib_right.npy gespeichert — 1024 Paare, range [0, 255]
